In this notebook, we aim to:
- generate candidate examples for all five tasks,
- inspect the generated examples,
- run simple quality checks,
- manually approve or reject examples,
- export the reviewed candidate pool,
- select approved base examples for the next phase.

In [4]:
import sys
from pathlib import Path
import json
import pandas as pd


In [ ]:
# Make sure the project root is on the Python path for imports. This allows us to import from src/ and other directories regardless of whether we're running from the root or the notebooks/ directory.
if str(Path.cwd().parent) not in sys.path and Path.cwd().name == "notebooks":
    sys.path.append(str(Path.cwd().parent))
elif str(Path.cwd()) not in sys.path and Path.cwd().name != "notebooks":
    sys.path.append(str(Path.cwd()))

In [5]:
from src.generation import (
    generate_single_label_candidates,
    generate_multi_label_candidates,
    generate_ie_candidates,
    generate_transformation_candidates,
    generate_qa_candidates,
    generate_all_candidates,
    render_input_for_review,
    candidate_to_review_row,
    auto_flag_candidate,
    find_exact_duplicate_inputs,
    set_review_status,
    get_examples_by_status,
    select_final_base_examples,
    example_to_dict,
    save_candidates_to_jsonl,
)

In [6]:
# Generate candidate examples for each task
single_label_candidates = generate_single_label_candidates()
multi_label_candidates = generate_multi_label_candidates()
ie_candidates = generate_ie_candidates()
transformation_candidates = generate_transformation_candidates()
qa_candidates = generate_qa_candidates()

print("Single-label candidates:", len(single_label_candidates))
print("Multi-label candidates:", len(multi_label_candidates))
print("Information extraction candidates:", len(ie_candidates))
print("Transformation candidates:", len(transformation_candidates))
print("Extractive QA candidates:", len(qa_candidates))

Single-label candidates: 36
Multi-label candidates: 48
Information extraction candidates: 40
Transformation candidates: 48
Extractive QA candidates: 40


In [7]:
# Combine all candidate pools
all_candidates = generate_all_candidates()
print("Total candidates:", len(all_candidates))

Total candidates: 212


In [11]:
# View one example from each task

task_examples = {}

for ex in all_candidates:
    if ex.task_name not in task_examples:
        task_examples[ex.task_name] = ex

for task_name, ex in task_examples.items():
    print("=" * 100)
    print("TASK:", task_name)
    print("EXAMPLE ID:", ex.example_id)
    print("TEMPLATE:", ex.template_name)
    print("INSTRUCTION:", ex.instruction)
    print("INPUT:")
    print(render_input_for_review(ex))
    print("GOLD OUTPUT:", ex.gold_output)
    print()

TASK: single_label_classification
EXAMPLE ID: slc_000
TEMPLATE: product_review
INSTRUCTION: Classify the sentiment of the text using exactly one label from {positive, negative, neutral}.
INPUT:
The fitness watch was excellent and worked smooth and satisfying.
GOLD OUTPUT: {'label': 'positive'}

TASK: multi_label_classification
EXAMPLE ID: mlc_036
TEMPLATE: institutional_announcement
INSTRUCTION: Assign all applicable topic labels from {finance, health, politics, sports, tech}. Return them alphabetically.
INPUT:
The government announced a new government regulation related to election reform.
GOLD OUTPUT: {'labels': ['politics']}

TASK: information_extraction
EXAMPLE ID: ie_084
TEMPLATE: meeting_announcement
INSTRUCTION: Extract the fields person, date, and location from the text.
INPUT:
Alice Smith will speak in Rome on 2024-01-15.
GOLD OUTPUT: {'person': 'Alice Smith', 'date': '2024-01-15', 'location': 'Rome'}

TASK: rule_based_transformation
EXAMPLE ID: rbt_124
TEMPLATE: mixed_case_st

In [ ]:
# Convert candidates to a review table 

review_rows = [candidate_to_review_row(ex) for ex in all_candidates]
review_df = pd.DataFrame(review_rows)

print("Review table shape:", review_df.shape)
review_df.head(10)

Review table shape: (212, 8)


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
0,slc_000,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was excellent and worked smo...,"{""label"": ""positive""}",pending,
1,slc_001,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was great and worked plea...,"{""label"": ""positive""}",pending,
2,slc_002,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was reliable and worked bette...,"{""label"": ""positive""}",pending,
3,slc_003,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was terrible and worked frust...,"{""label"": ""negative""}",pending,
4,slc_004,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was awful and worked worse t...,"{""label"": ""negative""}",pending,
5,slc_005,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was poor and worked annoying ...,"{""label"": ""negative""}",pending,
6,slc_006,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was ordinary and worked accep...,"{""label"": ""neutral""}",pending,
7,slc_007,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was fine and worked neither g...,"{""label"": ""neutral""}",pending,
8,slc_008,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was average and worked mo...,"{""label"": ""neutral""}",pending,
9,slc_009,single_label_classification,service_feedback,Classify the sentiment of the text using exact...,The staff at the store were excellent and the ...,"{""label"": ""positive""}",pending,


In [17]:
# Inspect a few examples from each task

for task_name in review_df["task_name"].unique():
    print("- " + task_name)
    display(review_df[review_df["task_name"] == task_name].head(5))

- single_label_classification


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
0,slc_000,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was excellent and worked smo...,"{""label"": ""positive""}",pending,
1,slc_001,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was great and worked plea...,"{""label"": ""positive""}",pending,
2,slc_002,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was reliable and worked bette...,"{""label"": ""positive""}",pending,
3,slc_003,single_label_classification,product_review,Classify the sentiment of the text using exact...,The budget phone was terrible and worked frust...,"{""label"": ""negative""}",pending,
4,slc_004,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was awful and worked worse t...,"{""label"": ""negative""}",pending,


- multi_label_classification


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
36,mlc_036,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new government regu...,"{""labels"": [""politics""]}",pending,
37,mlc_037,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The startup announced a new data pipeline rela...,"{""labels"": [""tech""]}",pending,
38,mlc_038,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The government announced a new nutrition study...,"{""labels"": [""health""]}",pending,
39,mlc_039,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The startup announced a new swimming related t...,"{""labels"": [""sports""]}",pending,
40,mlc_040,multi_label_classification,institutional_announcement,Assign all applicable topic labels from {finan...,The city council announced a new budget plan r...,"{""labels"": [""finance""]}",pending,


- information_extraction


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
84,ie_084,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Alice Smith will speak in Rome on 2024-01-15.,"{""person"": ""Alice Smith"", ""date"": ""2024-01-15""...",pending,
85,ie_085,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Bruno Rossi will speak in Milan on 2024-03-21.,"{""person"": ""Bruno Rossi"", ""date"": ""2024-03-21""...",pending,
86,ie_086,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Carla Gomez will speak in Paris on 2024-06-10.,"{""person"": ""Carla Gomez"", ""date"": ""2024-06-10""...",pending,
87,ie_087,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",David Lee will speak in Berlin on 2024-09-05.,"{""person"": ""David Lee"", ""date"": ""2024-09-05"", ...",pending,
88,ie_088,information_extraction,meeting_announcement,"Extract the fields person, date, and location ...",Elena Marino will speak in Madrid on 2025-02-14.,"{""person"": ""Elena Marino"", ""date"": ""2025-02-14...",pending,


- rule_based_transformation


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
124,rbt_124,rule_based_transformation,mixed_case_statement__lowercase,Convert the text to lowercase.,Today Farah Khan Visited Vienna With 3 Friends!,"{""text"": ""today farah khan visited vienna with...",pending,
125,rbt_125,rule_based_transformation,mixed_case_statement__lowercase,Convert the text to lowercase.,Today Elena Marino Visited Vienna With 7 Friends!,"{""text"": ""today elena marino visited vienna wi...",pending,
126,rbt_126,rule_based_transformation,mixed_case_statement__lowercase,Convert the text to lowercase.,Today Elena Marino Visited Milan With 10 Friends!,"{""text"": ""today elena marino visited milan wit...",pending,
127,rbt_127,rule_based_transformation,mixed_case_statement__remove_punctuation,Remove all punctuation from the text.,Today Hassan Ali Visited Rome With 3 Friends!,"{""text"": ""Today Hassan Ali Visited Rome With 3...",pending,
128,rbt_128,rule_based_transformation,mixed_case_statement__remove_punctuation,Remove all punctuation from the text.,Today Irene Novak Visited Rome With 7 Friends!,"{""text"": ""Today Irene Novak Visited Rome With ...",pending,


- extractive_qa


,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
172,qa_172,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Alice Smith arrived in Rome on 2024-0...,"{""answer"": ""Rome""}",pending,
173,qa_173,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Bruno Rossi arrived in Milan on 2024-...,"{""answer"": ""Milan""}",pending,
174,qa_174,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Carla Gomez arrived in Paris on 2024-...,"{""answer"": ""Paris""}",pending,
175,qa_175,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: David Lee arrived in Berlin on 2024-0...,"{""answer"": ""Berlin""}",pending,
176,qa_176,extractive_qa,schedule_passage,Answer the question using an exact span from t...,PASSAGE: Elena Marino arrived in Madrid on 202...,"{""answer"": ""Madrid""}",pending,


In [18]:
# Run automatic quality flags (These flags do not reject examples automatically, they only help identify suspicious cases for manual review)

flagged_rows = []

for ex in all_candidates:
    flags = auto_flag_candidate(ex)
    if flags:
        flagged_rows.append({
        "example_id": ex.example_id,
        "task_name": ex.task_name,
        "template_name": ex.template_name,
        "flags": ", ".join(flags),
        "rendered_input": render_input_for_review(ex),
        "gold_output": json.dumps(ex.gold_output, ensure_ascii=False)
        })

flagged_df = pd.DataFrame(flagged_rows)

print("Number of flagged examples:", len(flagged_df))
flagged_df.head(20)

Number of flagged examples: 0


""


In [19]:
# Check for exact duplicate inputs

duplicate_inputs = find_exact_duplicate_inputs(all_candidates)
print("Number of duplicate input strings:", len(duplicate_inputs))

Number of duplicate input strings: 1


In [20]:
duplicate_preview = [
{"rendered_input": text, "example_ids": ids}
for text, ids in list(duplicate_inputs.items())[:10]
]

duplicate_preview_df = pd.DataFrame(duplicate_preview)
duplicate_preview_df

,rendered_input,example_ids
0,Tiny words vanish slowly when elephantine expr...,"[rbt_160, rbt_161, rbt_162, rbt_163, rbt_164, ..."


In [21]:
# Approve a few examples manually as a demonstration

set_review_status(all_candidates, all_candidates[0].example_id, "approved", "Clear example.")
set_review_status(all_candidates, all_candidates[1].example_id, "approved", "Good wording.")
set_review_status(all_candidates, all_candidates[2].example_id, "rejected", "Too similar to another item.")

print("Approved:", len(get_examples_by_status(all_candidates, "approved")))
print("Rejected:", len(get_examples_by_status(all_candidates, "rejected")))
print("Pending:", len(get_examples_by_status(all_candidates, "pending")))

Approved: 2
Rejected: 1
Pending: 209


In [22]:
# Inspect reviewed examples

reviewed_rows = [candidate_to_review_row(ex) for ex in all_candidates]
reviewed_df = pd.DataFrame(reviewed_rows)

display(reviewed_df[reviewed_df["review_status"] != "pending"].head(20))

,example_id,task_name,template_name,instruction,rendered_input,gold_output,review_status,review_note
0,slc_000,single_label_classification,product_review,Classify the sentiment of the text using exact...,The fitness watch was excellent and worked smo...,"{""label"": ""positive""}",approved,Clear example.
1,slc_001,single_label_classification,product_review,Classify the sentiment of the text using exact...,The medical software was great and worked plea...,"{""label"": ""positive""}",approved,Good wording.
2,slc_002,single_label_classification,product_review,Classify the sentiment of the text using exact...,The robot vacuum was reliable and worked bette...,"{""label"": ""positive""}",rejected,Too similar to another item.


In [23]:
# Save the full candidate pool 

save_candidates_to_jsonl(all_candidates, "data/candidates/candidates.jsonl")
print("Saved candidate pool to data/candidates/candidates.jsonl")

Saved candidate pool to data/candidates/candidates.jsonl
